# TATN: Panasonic 25 C to CMU eVTOL

This notebook runs the same CMU core-seven protocol twice: (1) a TATN source trained once from the paper's Panasonic 25 C procedure, and (2) the authors' published `pre-25.pt`. CMU SOC references are used only after adaptation for evaluation.

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from urllib.parse import urlunsplit

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = urlunsplit(('https', 'github.com', '/zhangcastle/battery-soc-transfer-benchmark.git', '', ''))
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add the GITHUB_TOKEN Colab Secret and enable Notebook access')
if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / 'git-askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    command = (['git', '-C', str(REPO), 'pull', '--ff-only'] if (REPO / '.git').exists() else ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
del github_token
if result.returncode:
    raise RuntimeError('Authenticated GitHub setup failed; check token permission')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import battery_benchmark
print('Repository ready:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Package:', battery_benchmark.__file__)

## Shared protocol

- Panasonic source: all nine 25 C profiles, sampled at 1 Hz, 90-step non-overlapping windows.
- Self-trained source: paper-layout tensors; Adam, lr 1e-3, batch 64, max 300 epochs with label-safe early stopping.
- Released source: exact `pre-25.pt` and its required released-repository tensor layout.
- CMU adaptation: the same chronological complete-mission prefix used by DARM, reaching at least 8,778 seconds.
- Each CMU cell starts from the unchanged source checkpoint. No adapted state crosses cells.
- CMU target normalization uses only the Panasonic 25 C source ranges.

In [ ]:
import pandas as pd
import torch
from IPython.display import display
from battery_benchmark.utils.config import load_yaml

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
DRIVE_OUTPUT = Path(PATHS['output_root'])
PANASONIC_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
CMU_BASE_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1'
CMU_CONDITION_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4'
CMU_MANIFEST = REPO / 'data/manifests/cmu_evtol/cmu_evtol_supervised_budget_manifest_v0_1.csv'
OFFICIAL_PRE25 = DRIVE_OUTPUT / 'reproductions/shen_2023/tatn_layout_ablation/official_pre25/pre-25.pt'
RESULT_ROOT = DRIVE_OUTPUT / 'benchmarks/panasonic_to_cmu_evtol/tatn'
CELLS = ['VAH01', 'VAH17', 'VAH27', 'VAH30', 'VAH15', 'VAH12', 'VAH11']
SOURCE_EPOCHS, TRANSFER_EPOCHS, BATCH_SIZE = 300, 150, 64
for path in (PANASONIC_ROOT, CMU_BASE_ROOT, CMU_CONDITION_ROOT, CMU_MANIFEST, OFFICIAL_PRE25):
    assert path.exists(), f'Missing required input: {path}'

def run_variant(source_kind, output_name, released_checkpoint=None):
    command = [
        sys.executable, str(REPO / 'scripts/benchmarks/run_tatn_cmu.py'),
        '--panasonic-data-root', str(PANASONIC_ROOT),
        '--cmu-base-root', str(CMU_BASE_ROOT),
        '--cmu-condition-root', str(CMU_CONDITION_ROOT),
        '--cmu-manifest', str(CMU_MANIFEST),
        '--output-root', str(RESULT_ROOT / output_name),
        '--source-kind', source_kind, '--source-epochs', str(SOURCE_EPOCHS),
        '--transfer-epochs', str(TRANSFER_EPOCHS), '--batch-size', str(BATCH_SIZE),
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu', '--resume',
        '--cells', *CELLS,
    ]
    if released_checkpoint is not None:
        command += ['--released-checkpoint', str(released_checkpoint)]
    subprocess.run(command, check=True)
    result = pd.read_csv(RESULT_ROOT / output_name / 'condition_metrics.csv')
    display(result.pivot(index=['cell_id', 'condition'], columns='method', values=['overall_mae_percent', 'overall_rmse_percent']).round(3))
    return result

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('results:', RESULT_ROOT)

## Result 1 - source trained from the Panasonic 25 C paper protocol

In [ ]:
trained_results = run_variant('trained', 'trained_paper_layout')

## Result 2 - authors' published Panasonic 25 C checkpoint

In [ ]:
released_results = run_variant('released', 'released_pre25', OFFICIAL_PRE25)